# High-rise 04 - Dynamic response (HFPI / SDOF)

Read the Cp time series from notebook 02, build the per-floor Cf/Cm load
history, and run the building dynamic-response recipe: generalized modal
loads -> per-mode SDOF (RK45) integration -> per-floor displacements and
static-equivalent loads, then off-centre accelerations for comfort. Writes
response figures to `debug/` and a per-floor peak table to `deliverables/`.

The structural model (mode shapes, floor masses, natural frequencies) comes
from the case modes/floors/mode-shape CSVs when the `CFDMOD_HR_MODES_CSV` /
`_FLOORS_CSV` / `_MODE_SHAPE_CSVS` variables are set; otherwise a synthetic
cantilever sway + torsion model tuned to the case geometry is used so the
chain runs on the fixtures.

In [ ]:
import os
import pathlib

import numpy as np  # noqa: F401  (used across later cells)

import matplotlib

matplotlib.use("Agg")  # headless: notebooks write files, they do not display

from cfdmod import inflow_report, plot_config  # noqa: E402
from cfdmod.building import (  # noqa: E402
    BuildingCase,
    cf_per_floor,
    cm_per_floor,
    cp_from_pressure,
    example_building_case,
    example_building_structure,
    floor_accelerations,
    floor_load_source,
    peak_response_table,
    peak_value,
    plot_floor_mass,
    plot_mode_shape,
    plot_natural_frequencies,
    solve_building_response,
    structure_from_csvs,
)
from cfdmod.report import DebugWriter  # noqa: E402


def _find_repo(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    return start.resolve()


REPO = _find_repo(pathlib.Path.cwd())

plot_config.apply_style()

FIX = REPO / "fixtures" / "tests"
OUTPUT_BASE = pathlib.Path(
    os.environ.get("CFDMOD_HR_OUTPUT_BASE", REPO / "examples" / "high_rise" / "_run")
)
VERSION = os.environ.get("CFDMOD_HR_VERSION", "example")
print("REPO:", REPO)
print("OUTPUT_BASE:", OUTPUT_BASE, "| VERSION:", VERSION)

In [ ]:
from cfdmod.adapters.xdmf_h5 import XdmfH5Storage

# --- config -------------------------------------------------------------
MESH = pathlib.Path(
    os.environ.get("CFDMOD_HR_MESH", FIX / "pressure" / "galpao" / "galpao.normalized.lnas")
)
CASE_DATA = os.environ.get("CFDMOD_HR_CASE_DATA")
PARAMS = os.environ.get("CFDMOD_HR_PARAMS", "params_cat3.yaml")
if CASE_DATA:
    case = BuildingCase.from_case_data(pathlib.Path(CASE_DATA), PARAMS)
else:
    case = example_building_case(MESH)

DAMPING = float(os.environ.get("CFDMOD_HR_DAMPING", "0.02"))

ARTIFACTS = OUTPUT_BASE / "artifacts" / VERSION
cp = XdmfH5Storage(ARTIFACTS).read_data_source(pathlib.Path("cp.time_series"))
print(f"cp: {cp.n_elements} elements, {cp.time.n_timesteps} steps | {case.n_floors} floors")

In [ ]:
# --- per-floor load history + structural model --------------------------
cf = cf_per_floor(cp, str(MESH), case, directions=("x", "y"))
cm = cm_per_floor(cp, str(MESH), case, directions=("z",))
load = floor_load_source(cf, cm, case)

# Real structural data from CSVs, or a synthetic model for the fixtures.
MODES_CSV = os.environ.get("CFDMOD_HR_MODES_CSV")
FLOORS_CSV = os.environ.get("CFDMOD_HR_FLOORS_CSV")
SHAPE_CSVS = os.environ.get("CFDMOD_HR_MODE_SHAPE_CSVS")  # comma-separated, one per mode
if MODES_CSV and FLOORS_CSV and SHAPE_CSVS:
    structure = structure_from_csvs(MODES_CSV, FLOORS_CSV, SHAPE_CSVS.split(","))
else:
    structure = example_building_structure(case, load.n_elements)

response = solve_building_response(load, structure, damping_ratio=DAMPING)
acc = floor_accelerations(response, structure, point=(1.0, 0.0))
freqs_hz = np.asarray(structure.natural_frequencies) / (2 * np.pi)
print(f"{structure.n_modes} modes at {np.round(freqs_hz, 3)} Hz | damping {DAMPING}")

In [ ]:
from cfdmod.dynamics import plotting as dyn

# --- response figures + per-floor peak table ----------------------------
dbg = DebugWriter(OUTPUT_BASE, stage="dynamic", version=VERSION)

fig, _ = dyn.plot_force_spectrum(load, freqs_hz)
dbg.savefig(fig, "force_spectrum.png")
plot_config.close(fig)

top = load.n_elements - 1
disp_top = np.asarray(response.fields.read("disp_x"))[top]
plim = float(1.5 * max(np.abs(disp_top).max(), 1e-9))
fig, _ = dyn.plot_displacement(response, floor=top, plot_limit=plim)
dbg.savefig(fig, "top_floor_hodograph.png")
plot_config.close(fig)

table = peak_response_table(response, acc, case)
fig, ax = plot_config.new_axes(xlabel="peak displacement [m]", ylabel="z [m]", title="Per-floor peak displacement")
ax.plot(table["disp_x_peak"], table["z_mid"], "-o", ms=3, label="disp_x")
ax.plot(table["disp_y_peak"], table["z_mid"], "-o", ms=3, label="disp_y")
ax.legend()
dbg.savefig(fig, "peak_displacement.png")
plot_config.close(fig)

fig, _ = dyn.plot_acceleration_floor_by_floor(
    table["acc_mag_peak"].to_numpy(), float(freqs_hz[0]), rec_period=10
)
dbg.savefig(fig, "peak_acceleration.png")
plot_config.close(fig)

dbg.save_csv(table, "dynamic_response.csv", deliverable=True)
table

In [ ]:
# --- structural model figures + peak-acceleration methods ---------------
# Visualise the structural model that drove the response ...
fig, _ = plot_mode_shape(structure, 0, rotation_scale=structure.floors_radius)
dbg.savefig(fig, "mode_0.png")
plot_config.close(fig)
fig, _ = plot_floor_mass(structure)
dbg.savefig(fig, "floor_mass.png")
plot_config.close(fig)
fig, _ = plot_natural_frequencies(structure)
dbg.savefig(fig, "natural_frequencies.png")
plot_config.close(fig)

# ... and compare peak-estimation methods on the top-floor acceleration.
acc_top = np.asarray(acc.fields.read("acc_mag"))[top]
peak_methods = {
    m: peak_value(acc_top, m, f0=float(freqs_hz[0]))
    for m in ("max", "peak-factor", "gumbel")
}
print("top-floor peak acc [m/s^2]:", {k: round(v, 4) for k, v in peak_methods.items()})